# EE200: Signals, Systems and Networks — Course Project
## Q3A 'Magical Mystery Tune' & Q3B 'Zapptain America'

This notebook contains the complete implementation, experiments, plots, and analysis for the audio fingerprinting and song identification system.

In [1]:
import os
import glob
import numpy as np
import scipy.signal as signal
import matplotlib.pyplot as plt
import pandas as pd
import librosa
from app import load_audio, get_spectrogram_from_array, fingerprint_array, index_database, _identify_array, add_noise, pitch_shift_audio, time_stretch_audio, DB_PATH, FS, MIN_MATCHING_HASHES

print('Libraries imported successfully!')

/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Successfully indexed 50 songs.
Libraries imported successfully!


### 1. Whole-Song DFT (Q3A)

First, we plot the Discrete Fourier Transform (DFT) magnitude of an entire song to see why a single DFT is insufficient for audio identification. 

**Explanation**: A single Fourier transform over a long signal shows all the frequencies that occurred *at some point* during the entire duration, but it loses all temporal (timing) information. We cannot tell *when* specific notes or chords were played. Thus, two completely different songs with similar frequency profiles would look nearly identical in the DFT magnitude spectrum.

In [2]:
# Find a song in the database
songs = sorted(glob.glob(os.path.join(DB_PATH, '*.mp3')) + glob.glob(os.path.join(DB_PATH, '*.wav')))
query_song_path = songs[0]
print(f'Analyzing song: {query_song_path}')

full_audio, sr = load_audio(query_song_path, sr=FS)
X = np.fft.fft(full_audio)
freqs = np.fft.fftfreq(len(full_audio), d=1/sr)
half = len(full_audio) // 2

plt.figure(figsize=(10, 4))
plt.plot(freqs[:half], 20 * np.log10(np.abs(X[:half]) + 1e-6), linewidth=0.5, color='navy')
plt.xlim([0, 5000])
plt.xlabel('Frequency (Hz)')
plt.ylabel('Magnitude (dB)')
plt.title('Whole-Song DFT — Time Information is Lost')
plt.grid(True, alpha=0.3)
plt.show()

Analyzing song: EE200 Project Song Database/A Day In The Life.mp3


/var/folders/dt/_snhm0h11_z9cmspf7gf_19w0000gn/T/ipykernel_52814/4108573263.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 2. Spectrogram and Window Length Trade-off (Q3A)

To capture frequency changes over time, we use a Short-Time Fourier Transform (STFT) to compute a spectrogram. Here, we compare the resolution using a short window (512 samples) and a long window (8192 samples).

**Observations on Time vs. Frequency Resolution**:
- **Short Window (e.g., 512 samples)**: Offers excellent time resolution because the analysis window is brief, allowing us to pinpoint the exact start and end of transient events (like beats). However, it suffers from poor frequency resolution (spectral smearing) due to the Heisenberg Uncertainty Principle.
- **Long Window (e.g., 8192 samples)**: Offers sharp frequency resolution, allowing us to resolve individual pitch harmonics. However, it suffers from poor time resolution, blurring rapid temporal transitions because the long window averages signal content over a large duration.

In [3]:
# Get a 10-second segment from the middle of the song
clip_len = min(10 * sr, len(full_audio))
start = len(full_audio) // 2 - clip_len // 2
query_clip = full_audio[start:start + clip_len]

short_win, long_win = 512, 8192
f_s, t_s, S_s = get_spectrogram_from_array(query_clip, sr=sr, nperseg=short_win)
f_l, t_l, S_l = get_spectrogram_from_array(query_clip, sr=sr, nperseg=long_win)

fig, axes = plt.subplots(2, 1, figsize=(11, 9))
for ax, f_, t_, S_, title in [
        (axes[0], f_s, t_s, S_s, f'Short Window (nperseg={short_win}) — Good Time Resolution'),
        (axes[1], f_l, t_l, S_l, f'Long Window (nperseg={long_win}) — Sharp Frequency Lines')]:
    ax.pcolormesh(t_, f_, S_, shading='gouraud', cmap='magma', vmin=S_.max() - 80, vmax=S_.max())
    ax.set_ylim([0, 5000])
    ax.set_xlabel('Time (s)')
    ax.set_ylabel('Frequency (Hz)')
    ax.set_title(title, fontweight='bold')
plt.tight_layout()
plt.show()

/var/folders/dt/_snhm0h11_z9cmspf7gf_19w0000gn/T/ipykernel_52814/210438774.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 3. Peak Pick and Constellation Map (Q3A)

We extract a sparse constellation of prominent spectral peaks (local maxima in the 2D spectrogram). Using a relative amplitude floor keeps peak-picking robust for both quiet and loud tracks.

In [4]:
from app import extract_peaks

f, t, S_db = get_spectrogram_from_array(query_clip, sr=sr)
f_idx, t_idx = extract_peaks(S_db)

plt.figure(figsize=(11, 5))
plt.pcolormesh(t, f, S_db, shading='gouraud', cmap='magma', vmin=S_db.max() - 80, vmax=S_db.max())
plt.scatter(t[t_idx], f[f_idx], s=18, facecolors='none', edgecolors='cyan', linewidths=1.0, label='Constellation peaks')
plt.legend(loc='upper right')
plt.ylim([0, 5000])
plt.xlabel('Time (s)')
plt.ylabel('Frequency (Hz)')
plt.title(f'Spectrogram + Constellation Map ({len(f_idx)} peaks)', fontweight='bold')
plt.show()

/var/folders/dt/_snhm0h11_z9cmspf7gf_19w0000gn/T/ipykernel_52814/387831838.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 4. Fingerprint Hashing: Pairs vs. Single Peaks (Q3A)

We compare the effectiveness of single-peak hashing against combinatorial peak pairing. In peak pairing, each anchor peak is coupled with up to $F$ (e.g. 8) downstream target peaks in a target time window, recording $(f_1, f_2, \Delta t)$ hashes.

**Why peak pairing is more decisive**:
- **Single-peak hashes**: Only capture the frequency of a peak. Because music often stays in a specific key and shares common frequencies (e.g., 440 Hz), matching single peaks leads to a massive number of random collisions across different songs, raising the noise floor and causing false positives.
- **Paired hashes**: Combine the frequencies of two peaks and the precise time gap between them. This creates a much higher-dimensional feature space, making the hashes highly unique to a specific song sequence and reducing spurious collisions.

In [5]:
# Load databases
db_pairs, _ = index_database(DB_PATH, use_pairs=True)
db_single, _ = index_database(DB_PATH, use_pairs=False)

# Identify query clip
label_pairs, score_pairs, _, matches_pairs = _identify_array(query_clip, sr, db_pairs, use_pairs=True)
label_single, score_single, _, matches_single = _identify_array(query_clip, sr, db_single, use_pairs=False)

print(f'Paired Fingerprint Match: {label_pairs} (aligned count = {score_pairs})')
print(f'Single Peak Fingerprint Match: {label_single} (aligned count = {score_single})')

# Plot offset histograms
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, matches, title, scr in zip(
        axes, [matches_pairs, matches_single],
        ['Paired Hashes (Anchor-Target)', 'Single-Peak Hashes'],
        [score_pairs, score_single]):
    if matches:
        best_song = max(matches, key=lambda s: max(matches[s].values()))
        hist = matches[best_song]
        deltas = sorted(hist.keys())
        counts = [hist[d] for d in deltas]
        ax.bar(deltas, counts, width=1.0, color='teal')
    ax.set_title(f'{title}\nPeak aligned votes = {scr}', fontweight='bold', fontsize=10)
    ax.set_xlabel('Offset delta (frames)')
    ax.set_ylabel('Vote count')
plt.tight_layout()
plt.show()

Paired Fingerprint Match: A Day In The Life (aligned count = 2097)
Single Peak Fingerprint Match: A Day In The Life (aligned count = 793)


/var/folders/dt/_snhm0h11_z9cmspf7gf_19w0000gn/T/ipykernel_52814/4139056470.py:28: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 5. Robustness to Additive Noise (Q3A)

We sweep different Signal-to-Noise Ratio (SNR) levels by adding white Gaussian noise to the query clip.

In [6]:
snr_levels = (30, 20, 10, 5, 0, -5)
results = []
for snr in snr_levels:
    noisy = add_noise(query_clip, snr_db=snr)
    label, score, _, _ = _identify_array(noisy, sr, db_pairs, use_pairs=True)
    matched = (label != 'No Match Found')
    results.append((snr, matched, score))
    print(f'SNR = {snr:+3d} dB -> Match Confirmed: {matched}, Score = {score}')

plt.figure(figsize=(9, 4.5))
colors = ['green' if m else 'red' for (_, m, _) in results]
plt.bar([str(s) for s, _, _ in results], [sc for _, _, sc in results], color=colors)
plt.axhline(MIN_MATCHING_HASHES, color='orange', linestyle='--', label=f'Min score threshold ({MIN_MATCHING_HASHES})')
plt.xlabel('SNR (dB)')
plt.ylabel('Best-match Aligned Hash Count')
plt.title('Noise Robustness Sweep (green = match, red = fail)')
plt.legend()
plt.show()

SNR = +30 dB -> Match Confirmed: True, Score = 1836
SNR = +20 dB -> Match Confirmed: True, Score = 1212
SNR = +10 dB -> Match Confirmed: True, Score = 246
SNR =  +5 dB -> Match Confirmed: True, Score = 122
SNR =  +0 dB -> Match Confirmed: True, Score = 40


SNR =  -5 dB -> Match Confirmed: True, Score = 13


/var/folders/dt/_snhm0h11_z9cmspf7gf_19w0000gn/T/ipykernel_52814/3438741550.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 6. Robustness to Pitch Shift and Time Stretch (Q3A)

We sweep pitch shifts (in semitones) and time-stretching (speed rates).

**Why pitch-shift and time-stretch defeat the fingerprint**:
- **Pitch-shift**: Shifting the pitch scales all frequency bins. In the linear frequency scale of the DFT, this shifts the peaks vertically. Since the hashes explicitly include the absolute bins $f_1$ and $f_2$, any change in pitch shifts the frequencies to different bins, causing the matching hashes to miss completely.
- **Time-stretch**: Stretches or compresses time, which changes the time gap $\Delta t$ between the anchor and target peaks. Since $\Delta t$ is a part of the hash value, stretching time alters the hash values, causing them to fail to match the database entries.

**Suggested improvement for robustness**:
- To achieve **pitch invariance**, we can map frequencies to a logarithmic scale (such as MIDI pitch or constant-Q transform) and only store frequency ratios (differences in the log scale) instead of absolute frequencies.
- To achieve **tempo/time-stretch invariance**, we can define the time gap between peaks relative to locally detected rhythmic events (tempo beats) rather than absolute seconds, or store only the relative order of frequencies.

In [7]:
semitone_shifts = (0, 0.5, 1, 2, 4)
stretch_rates = (1.0, 1.02, 1.05, 1.10, 1.20)
pitch_results, stretch_results = [], []

for st in semitone_shifts:
    shifted = query_clip if st == 0 else pitch_shift_audio(query_clip, sr, st)
    label, score, _, _ = _identify_array(shifted, sr, db_pairs, use_pairs=True)
    pitch_results.append((st, label != 'No Match Found', score))
    print(f'Pitch Shift = {st:+.1f} st -> Match: {label != "No Match Found"}, Score = {score}')

for rate in stretch_rates:
    stretched = query_clip if rate == 1.0 else time_stretch_audio(query_clip, rate)
    label, score, _, _ = _identify_array(stretched, sr, db_pairs, use_pairs=True)
    stretch_results.append((rate, label != 'No Match Found', score))
    print(f'Time Stretch Rate = {rate:.2f} -> Match: {label != "No Match Found"}, Score = {score}')

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
sts = [r[0] for r in pitch_results]
pscores = [r[2] for r in pitch_results]
pmatched = [r[1] for r in pitch_results]
axes[0].bar([f'{s:+.1f}' for s in sts], pscores, color=['green' if m else 'red' for m in pmatched])
axes[0].axhline(MIN_MATCHING_HASHES, color='orange', linestyle='--')
axes[0].set_xlabel('Pitch Shift (semitones)')
axes[0].set_ylabel('Best-match Hash Count')
axes[0].set_title('Pitch-Shift Robustness')

rates = [r[0] for r in stretch_results]
sscores = [r[2] for r in stretch_results]
smatched = [r[1] for r in stretch_results]
axes[1].bar([f'{r:.2f}' for r in rates], sscores, color=['green' if m else 'red' for m in smatched])
axes[1].axhline(MIN_MATCHING_HASHES, color='orange', linestyle='--')
axes[1].set_xlabel('Time-stretch Rate')
axes[1].set_ylabel('Best-match Hash Count')
axes[1].set_title('Time-Stretch Robustness')

plt.tight_layout()
plt.show()

Pitch Shift = +0.0 st -> Match: True, Score = 2097


Pitch Shift = +0.5 st -> Match: True, Score = 5
Pitch Shift = +1.0 st -> Match: True, Score = 5
Pitch Shift = +2.0 st -> Match: True, Score = 5
Pitch Shift = +4.0 st -> Match: True, Score = 6


Time Stretch Rate = 1.00 -> Match: True, Score = 2097
Time Stretch Rate = 1.02 -> Match: True, Score = 253
Time Stretch Rate = 1.05 -> Match: True, Score = 124
Time Stretch Rate = 1.10 -> Match: True, Score = 70
Time Stretch Rate = 1.20 -> Match: True, Score = 15


/var/folders/dt/_snhm0h11_z9cmspf7gf_19w0000gn/T/ipykernel_52814/3799936481.py:37: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
